# Experiment 03 — Compact 1D CNN Sequence Models

## Purpose and decision rules

The engineered-feature Random Forest is the current classical benchmark:

**mean subject-fold macro-F1: 0.7257 ± 0.0449**

This experiment asks whether a compact neural model can learn useful temporal
structure directly from the 20 Hz watch sequences.

Two predefined architectures will be compared:

1. an accelerometer-only 1D CNN;
2. a dual-branch accelerometer–gyroscope 1D CNN.

Both models use the same paired examples, outer subject folds, training
procedure, and evaluation metrics. Inputs contain only the three physical axes;
coverage masks are retained for provenance but are not model channels.

### Nested training protocol

For every outer subject fold:

1. reserve the outer fold exclusively for model evaluation;
2. split the remaining subjects deterministically into inner-training and
   inner-validation groups;
3. fit per-sensor, per-axis normalization using inner-training subjects only;
4. use inner-validation macro-F1 for early stopping and epoch selection;
5. reinitialize the model;
6. refit normalization using all outer-training subjects;
7. train for the selected number of epochs on all outer-training examples;
8. evaluate once on the untouched outer-fold subjects.

The outer-fold labels are never used for normalization, early stopping, epoch
selection, or other training decisions.

### Comparison rules

The dual-branch model is considered meaningfully better than the
accelerometer-only CNN if it improves mean macro-F1 by at least 0.01 and
improves at least three of five folds.

A CNN is considered a practically meaningful improvement over the engineered
Random Forest if it reaches mean macro-F1 of at least **0.7357** and improves
at least three folds relative to the Random Forest.

Architecture comparison begins with random seed 42. After selecting the
stronger CNN architecture, that architecture will be repeated across three
seeds before drawing a final conclusion.

No test artifact is loaded or evaluated.

In [1]:
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch


print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

Python: 3.12.1
PyTorch: 2.13.0
NumPy: 2.5.2
pandas: 3.0.5
scikit-learn: 1.9.0
MPS available: True
CUDA available: False


## 1. Load development sequences and define nested subject splits

Only the paired-watch development artifact is loaded.

For each outer fold, the next numbered fold is used as inner validation:

| Outer fold | Inner-validation fold |
|---:|---:|
| 1 | 2 |
| 2 | 3 |
| 3 | 4 |
| 4 | 5 |
| 5 | 1 |

The other three folds form inner training. After selecting an epoch using this
inner split, the model will be reinitialized and trained on all four
outer-training folds.

This rotation is deterministic, reuses the established subject groups, and
keeps inner training, inner validation, and outer validation subject-disjoint
within every run.

In [2]:
import json


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate
        for candidate in [
            CURRENT_DIR,
            *CURRENT_DIR.parents,
        ]
        if (
            candidate
            / "modified"
            / "watch_pair_20hz"
            / "preprocessing_meta.json"
        ).exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root."
    )

ARTIFACT_DIR = (
    PROJECT_ROOT
    / "modified"
    / "watch_pair_20hz"
)

TRAIN_SEQUENCE_PATH = (
    ARTIFACT_DIR / "train_sequences.npz"
)
TRAIN_METADATA_PATH = (
    ARTIFACT_DIR / "train_metadata.parquet"
)
PREPROCESSING_META_PATH = (
    ARTIFACT_DIR / "preprocessing_meta.json"
)

preprocessing_meta = json.loads(
    PREPROCESSING_META_PATH.read_text(
        encoding="utf-8"
    )
)

with np.load(TRAIN_SEQUENCE_PATH) as artifact:
    train_accel = artifact[
        "accel"
    ].copy()
    train_gyro = artifact[
        "gyro"
    ].copy()

train_metadata = pd.read_parquet(
    TRAIN_METADATA_PATH
)

assert train_accel.shape == (
    25_585,
    100,
    3,
)
assert train_gyro.shape == train_accel.shape

assert np.array_equal(
    train_metadata[
        "array_index"
    ].to_numpy(),
    np.arange(len(train_metadata)),
)

ACTIVITY_CODES = preprocessing_meta[
    "labels"
]["activity_codes_in_order"]

ACTIVITY_TO_INDEX = {
    activity_code: activity_index
    for activity_index, activity_code
    in enumerate(ACTIVITY_CODES)
}

labels = (
    train_metadata[
        "activity_code"
    ]
    .astype(str)
    .map(ACTIVITY_TO_INDEX)
    .to_numpy(dtype=np.int64)
)

fold_assignments = (
    train_metadata["fold"]
    .astype(int)
    .to_numpy()
)

subject_ids = train_metadata[
    "user_id"
].to_numpy()

DEVICE = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(
    f"Loaded {len(train_metadata):,} examples."
)
print("Sequence shape:", train_accel.shape)
print("Activities:", len(ACTIVITY_CODES))
print("Training device:", DEVICE)

Loaded 25,585 examples.
Sequence shape: (25585, 100, 3)
Activities: 18
Training device: mps


In [3]:
INNER_VALIDATION_FOLD_BY_OUTER = {
    1: 2,
    2: 3,
    3: 4,
    4: 5,
    5: 1,
}

nested_split_masks = {}
split_rows = []

for outer_fold, inner_validation_fold in (
    INNER_VALIDATION_FOLD_BY_OUTER.items()
):
    outer_validation_mask = (
        fold_assignments == outer_fold
    )

    outer_training_mask = (
        ~outer_validation_mask
    )

    inner_validation_mask = (
        fold_assignments
        == inner_validation_fold
    )

    inner_training_mask = (
        outer_training_mask
        & ~inner_validation_mask
    )

    inner_training_subjects = set(
        subject_ids[inner_training_mask]
    )
    inner_validation_subjects = set(
        subject_ids[inner_validation_mask]
    )
    outer_validation_subjects = set(
        subject_ids[outer_validation_mask]
    )

    assert inner_training_subjects.isdisjoint(
        inner_validation_subjects
    )
    assert inner_training_subjects.isdisjoint(
        outer_validation_subjects
    )
    assert inner_validation_subjects.isdisjoint(
        outer_validation_subjects
    )

    nested_split_masks[outer_fold] = {
        "outer_training": (
            outer_training_mask
        ),
        "outer_validation": (
            outer_validation_mask
        ),
        "inner_training": (
            inner_training_mask
        ),
        "inner_validation": (
            inner_validation_mask
        ),
    }

    split_rows.append({
        "outer_fold": outer_fold,
        "inner_validation_fold": (
            inner_validation_fold
        ),
        "inner_training_subjects": len(
            inner_training_subjects
        ),
        "inner_validation_subjects": len(
            inner_validation_subjects
        ),
        "outer_validation_subjects": len(
            outer_validation_subjects
        ),
        "inner_training_examples": int(
            inner_training_mask.sum()
        ),
        "inner_validation_examples": int(
            inner_validation_mask.sum()
        ),
        "outer_validation_examples": int(
            outer_validation_mask.sum()
        ),
        "inner_training_activities": int(
            np.unique(
                labels[inner_training_mask]
            ).size
        ),
        "inner_validation_activities": int(
            np.unique(
                labels[inner_validation_mask]
            ).size
        ),
    })

nested_split_overview = pd.DataFrame(
    split_rows
)

display(nested_split_overview)

print("Nested subject splits constructed.")

,outer_fold,inner_validation_fold,inner_training_subjects,inner_validation_subjects,outer_validation_subjects,inner_training_examples,inner_validation_examples,outer_validation_examples,inner_training_activities,inner_validation_activities
0,1,2,24,8,9,14945,4970,5670,18,18
1,2,3,25,8,8,15645,4970,4970,18,18
2,3,4,25,8,8,15645,4970,4970,18,18
3,4,5,25,8,8,15610,5005,4970,18,18
4,5,1,24,9,8,14910,5670,5005,18,18


Nested subject splits constructed.


## 2. Fold-specific normalization and data loaders

Normalization is fitted separately for accelerometer and gyroscope using only
the applicable training partition.

For each sensor, one mean and standard deviation is calculated per physical
axis across all training examples and timesteps. The validation arrays are
transformed using those training-derived values.

PyTorch receives each sequence in channel-first form `(3, 100)`.

In [4]:
from torch.utils.data import (
    DataLoader,
    Dataset,
)


BATCH_SIZE = 256


def fit_sensor_normalization(
    sequences,
    training_mask,
):
    training_values = sequences[
        training_mask
    ].astype(
        np.float64,
        copy=False,
    )

    mean = training_values.mean(
        axis=(0, 1)
    )
    std = training_values.std(
        axis=(0, 1),
        ddof=0,
    )

    if np.any(
        std <= np.finfo(np.float32).eps
    ):
        raise ValueError(
            "At least one signal axis has "
            "near-zero training variance."
        )

    return {
        "mean": mean.astype(np.float32),
        "std": std.astype(np.float32),
    }


def fit_fold_normalization(
    training_mask,
):
    return {
        "accel": fit_sensor_normalization(
            train_accel,
            training_mask,
        ),
        "gyro": fit_sensor_normalization(
            train_gyro,
            training_mask,
        ),
    }

In [5]:
class PairedSequenceDataset(Dataset):
    def __init__(
        self,
        indices,
        accel_sequences,
        gyro_sequences,
        target_labels,
        normalization,
    ):
        self.indices = np.asarray(
            indices,
            dtype=np.int64,
        )
        self.accel_sequences = (
            accel_sequences
        )
        self.gyro_sequences = (
            gyro_sequences
        )
        self.target_labels = (
            target_labels
        )
        self.normalization = (
            normalization
        )

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        array_index = int(
            self.indices[position]
        )

        accel = (
            (
                self.accel_sequences[
                    array_index
                ]
                - self.normalization[
                    "accel"
                ]["mean"]
            )
            / self.normalization[
                "accel"
            ]["std"]
        )

        gyro = (
            (
                self.gyro_sequences[
                    array_index
                ]
                - self.normalization[
                    "gyro"
                ]["mean"]
            )
            / self.normalization[
                "gyro"
            ]["std"]
        )

        accel = np.ascontiguousarray(
            accel.T,
            dtype=np.float32,
        )
        gyro = np.ascontiguousarray(
            gyro.T,
            dtype=np.float32,
        )

        label = int(
            self.target_labels[
                array_index
            ]
        )

        return (
            torch.from_numpy(accel),
            torch.from_numpy(gyro),
            torch.tensor(
                label,
                dtype=torch.long,
            ),
            torch.tensor(
                array_index,
                dtype=torch.long,
            ),
        )


def make_data_loader(
    mask,
    normalization,
    *,
    shuffle,
    seed,
):
    indices = np.flatnonzero(mask)

    dataset = PairedSequenceDataset(
        indices=indices,
        accel_sequences=train_accel,
        gyro_sequences=train_gyro,
        target_labels=labels,
        normalization=normalization,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        drop_last=False,
        generator=generator,
    )

Perform one functional check:

In [6]:
example_masks = nested_split_masks[1]

example_normalization = (
    fit_fold_normalization(
        example_masks[
            "inner_training"
        ]
    )
)

example_loader = make_data_loader(
    example_masks[
        "inner_validation"
    ],
    example_normalization,
    shuffle=False,
    seed=42,
)

(
    example_accel,
    example_gyro,
    example_labels,
    example_indices,
) = next(iter(example_loader))

print(
    "Accelerometer batch:",
    tuple(example_accel.shape),
    example_accel.dtype,
)
print(
    "Gyroscope batch:",
    tuple(example_gyro.shape),
    example_gyro.dtype,
)
print(
    "Label batch:",
    tuple(example_labels.shape),
    example_labels.dtype,
)
print(
    "Index batch:",
    tuple(example_indices.shape),
    example_indices.dtype,
)

Accelerometer batch: (256, 3, 100) torch.float32
Gyroscope batch: (256, 3, 100) torch.float32
Label batch: (256,) torch.int64
Index batch: (256,) torch.int64


## 3. Fixed CNN architectures and training configuration

Both models use the same compact sensor branch:

1. 32 filters with kernel size 7;
2. max pooling by 2;
3. 64 filters with kernel size 5;
4. max pooling by 2;
5. 128 filters with kernel size 3;
6. global average pooling.

Each convolution is followed by batch normalization and ReLU activation.

The accelerometer-only model classifies the 128-dimensional accelerometer
embedding directly. The dual-branch model applies independent branches to
accelerometer and gyroscope, concatenates their embeddings, and uses one
128-unit fusion layer before classification.

Training is fixed before evaluation:

- cross-entropy loss without class weighting;
- AdamW optimizer;
- learning rate `0.001`;
- weight decay `0.0001`;
- batch size `256`;
- maximum 50 inner-training epochs;
- early-stopping patience of 6 epochs;
- minimum macro-F1 improvement of `0.001`;
- dropout probability `0.3`;
- initial comparison seed `42`.

No architecture or optimizer tuning is performed in this experiment.

In [7]:
from torch import nn


DROPOUT_RATE = 0.3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_INNER_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 6
EARLY_STOPPING_MIN_DELTA = 0.001
INITIAL_SEED = 42


class SensorBranch(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(
                in_channels=3,
                out_channels=32,
                kernel_size=7,
                padding=3,
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(
                in_channels=32,
                out_channels=64,
                kernel_size=5,
                padding=2,
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, sequence):
        encoded = self.encoder(sequence)
        return encoded.squeeze(-1)

In [8]:
class AccelerometerCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.accel_branch = SensorBranch()

        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(128, n_classes),
        )

    def forward(self, accel, gyro):
        del gyro

        accel_embedding = (
            self.accel_branch(accel)
        )

        return self.classifier(
            accel_embedding
        )


class DualBranchCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.accel_branch = SensorBranch()
        self.gyro_branch = SensorBranch()

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(128, n_classes),
        )

    def forward(self, accel, gyro):
        accel_embedding = (
            self.accel_branch(accel)
        )
        gyro_embedding = (
            self.gyro_branch(gyro)
        )

        fused_embedding = torch.cat(
            [
                accel_embedding,
                gyro_embedding,
            ],
            dim=1,
        )

        return self.classifier(
            fused_embedding
        )

Define the two fixed configurations and perform one forward pass:

In [9]:
MODEL_CONFIGS = {
    "accelerometer": {
        "label": "Accelerometer-only CNN",
        "builder": lambda: AccelerometerCNN(
            n_classes=len(ACTIVITY_CODES)
        ),
    },
    "dual_branch": {
        "label": (
            "Dual-branch accelerometer + gyroscope CNN"
        ),
        "builder": lambda: DualBranchCNN(
            n_classes=len(ACTIVITY_CODES)
        ),
    },
}


def count_trainable_parameters(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


architecture_rows = []

for model_key, config in MODEL_CONFIGS.items():
    model = config["builder"]().to(
        DEVICE
    )

    model.eval()

    with torch.no_grad():
        logits = model(
            example_accel.to(DEVICE),
            example_gyro.to(DEVICE),
        )

    assert logits.shape == (
        len(example_accel),
        len(ACTIVITY_CODES),
    )

    architecture_rows.append({
        "model_key": model_key,
        "model": config["label"],
        "trainable_parameters": (
            count_trainable_parameters(
                model
            )
        ),
        "output_shape": tuple(
            logits.shape
        ),
    })

    del model
    del logits

    if DEVICE.type == "mps":
        torch.mps.empty_cache()

architecture_overview = pd.DataFrame(
    architecture_rows
)

display(architecture_overview)

print("Both CNN forward passes succeeded.")

,model_key,model,trainable_parameters,output_shape
0,accelerometer,Accelerometer-only CNN,38482,"(256, 18)"
1,dual_branch,Dual-branch accelerometer + gyroscope CNN,107538,"(256, 18)"


Both CNN forward passes succeeded.


## 4. Shared training and inner epoch selection

The inner-validation partition is used only to select the training duration.
Macro-F1 is calculated after every epoch, and training stops after six epochs
without a qualifying improvement.

The selected epoch will later be used to train a newly initialized model on all
outer-training subjects.

In [10]:
from sklearn.metrics import f1_score


def set_training_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if DEVICE.type == "mps":
        torch.mps.manual_seed(seed)


def train_one_epoch(
    model,
    data_loader,
    optimizer,
    loss_function,
):
    model.train()

    total_loss = 0.0
    total_examples = 0

    for (
        accel,
        gyro,
        target,
        _,
    ) in data_loader:
        accel = accel.to(DEVICE)
        gyro = gyro.to(DEVICE)
        target = target.to(DEVICE)

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(accel, gyro)
        loss = loss_function(
            logits,
            target,
        )

        loss.backward()
        optimizer.step()

        batch_size = len(target)

        total_loss += (
            float(loss.detach().cpu())
            * batch_size
        )
        total_examples += batch_size

    return total_loss / total_examples


def predict_data_loader(
    model,
    data_loader,
):
    model.eval()

    true_parts = []
    predicted_parts = []
    index_parts = []

    with torch.no_grad():
        for (
            accel,
            gyro,
            target,
            array_index,
        ) in data_loader:
            accel = accel.to(DEVICE)
            gyro = gyro.to(DEVICE)

            logits = model(accel, gyro)

            predicted = logits.argmax(
                dim=1
            )

            true_parts.append(
                target.numpy()
            )
            predicted_parts.append(
                predicted.cpu().numpy()
            )
            index_parts.append(
                array_index.numpy()
            )

    return {
        "y_true": np.concatenate(
            true_parts
        ),
        "y_pred": np.concatenate(
            predicted_parts
        ),
        "array_index": np.concatenate(
            index_parts
        ),
    }


def calculate_macro_f1(
    y_true,
    y_pred,
):
    return f1_score(
        y_true,
        y_pred,
        labels=np.arange(
            len(ACTIVITY_CODES)
        ),
        average="macro",
        zero_division=0,
    )

In [11]:
def select_epoch_with_inner_validation(
    model_key,
    outer_fold,
    seed,
    *,
    verbose=True,
):
    masks = nested_split_masks[
        outer_fold
    ]

    normalization = (
        fit_fold_normalization(
            masks["inner_training"]
        )
    )

    training_loader = make_data_loader(
        masks["inner_training"],
        normalization,
        shuffle=True,
        seed=(
            seed
            + outer_fold * 100
        ),
    )

    validation_loader = make_data_loader(
        masks["inner_validation"],
        normalization,
        shuffle=False,
        seed=seed,
    )

    set_training_seed(seed)

    model = MODEL_CONFIGS[
        model_key
    ]["builder"]().to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    loss_function = (
        nn.CrossEntropyLoss()
    )

    history_rows = []

    best_macro_f1 = -np.inf
    best_epoch = None
    epochs_without_improvement = 0

    for epoch in range(
        1,
        MAX_INNER_EPOCHS + 1,
    ):
        training_loss = train_one_epoch(
            model,
            training_loader,
            optimizer,
            loss_function,
        )

        validation_predictions = (
            predict_data_loader(
                model,
                validation_loader,
            )
        )

        validation_macro_f1 = (
            calculate_macro_f1(
                validation_predictions[
                    "y_true"
                ],
                validation_predictions[
                    "y_pred"
                ],
            )
        )

        improved = (
            validation_macro_f1
            > best_macro_f1
            + EARLY_STOPPING_MIN_DELTA
        )

        if improved:
            best_macro_f1 = (
                validation_macro_f1
            )
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history_rows.append({
            "epoch": epoch,
            "training_loss": (
                training_loss
            ),
            "inner_validation_macro_f1": (
                validation_macro_f1
            ),
            "selected_improvement": (
                improved
            ),
        })

        if verbose and (
            improved
            or epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                f"Epoch {epoch:02d}: "
                f"loss={training_loss:.4f}, "
                f"inner macro-F1="
                f"{validation_macro_f1:.4f}"
            )

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            break

    result = {
        "model_key": model_key,
        "outer_fold": outer_fold,
        "inner_validation_fold": (
            INNER_VALIDATION_FOLD_BY_OUTER[
                outer_fold
            ]
        ),
        "seed": seed,
        "selected_epoch": int(
            best_epoch
        ),
        "best_inner_macro_f1": float(
            best_macro_f1
        ),
        "epochs_run": len(
            history_rows
        ),
        "history": pd.DataFrame(
            history_rows
        ),
    }

    del model
    del optimizer

    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    return result

First real epoch-selection job:

In [12]:
inner_selection_cache = {}

first_selection_key = (
    "accelerometer",
    1,
    INITIAL_SEED,
)

first_selection = (
    select_epoch_with_inner_validation(
        model_key="accelerometer",
        outer_fold=1,
        seed=INITIAL_SEED,
    )
)

inner_selection_cache[
    first_selection_key
] = first_selection

display(
    pd.Series({
        "Model": MODEL_CONFIGS[
            "accelerometer"
        ]["label"],
        "Outer fold": 1,
        "Inner-validation fold": (
            first_selection[
                "inner_validation_fold"
            ]
        ),
        "Selected epoch": (
            first_selection[
                "selected_epoch"
            ]
        ),
        "Best inner macro-F1": (
            first_selection[
                "best_inner_macro_f1"
            ]
        ),
        "Epochs run": (
            first_selection[
                "epochs_run"
            ]
        ),
    }).to_frame("value")
)

display(
    first_selection[
        "history"
    ].tail(10).round(4)
)

Epoch 01: loss=2.0048, inner macro-F1=0.4369
Epoch 02: loss=1.4699, inner macro-F1=0.5312
Epoch 03: loss=1.2257, inner macro-F1=0.5446
Epoch 04: loss=1.0980, inner macro-F1=0.5643
Epoch 05: loss=1.0260, inner macro-F1=0.5897
Epoch 08: loss=0.8859, inner macro-F1=0.5966
Epoch 11: loss=0.8179, inner macro-F1=0.6039
Epoch 12: loss=0.8001, inner macro-F1=0.6074
Epoch 15: loss=0.7597, inner macro-F1=0.6197
Epoch 21: loss=0.6824, inner macro-F1=0.6155


,value
Model,Accelerometer-only CNN
Outer fold,1
Inner-validation fold,2
Selected epoch,15
Best inner macro-F1,0.619672
Epochs run,21


,epoch,training_loss,inner_validation_macro_f1,selected_improvement
11,12,0.8001,0.6074,True
12,13,0.7827,0.5922,False
13,14,0.7760,0.6037,False
14,15,0.7597,0.6197,True
15,16,0.7262,0.6067,False
16,17,0.7193,0.5779,False
17,18,0.7146,0.6088,False
18,19,0.7065,0.6007,False
19,20,0.6862,0.6025,False
20,21,0.6824,0.6155,False


## 5. Nested outer-fold CNN comparison

For each model and outer fold, the inner split first selects the training
duration. A newly initialized model is then trained for that duration using all
outer-training subjects and normalization fitted on those subjects.

Each outer-validation example receives exactly one prediction. Outer-validation
performance does not affect training or epoch selection.

In [13]:
def train_outer_model(
    model_key,
    outer_fold,
    seed,
    selected_epochs,
):
    masks = nested_split_masks[
        outer_fold
    ]

    normalization = (
        fit_fold_normalization(
            masks["outer_training"]
        )
    )

    training_loader = make_data_loader(
        masks["outer_training"],
        normalization,
        shuffle=True,
        seed=(
            seed
            + outer_fold * 1_000
        ),
    )

    validation_loader = make_data_loader(
        masks["outer_validation"],
        normalization,
        shuffle=False,
        seed=seed,
    )

    set_training_seed(seed)

    model = MODEL_CONFIGS[
        model_key
    ]["builder"]().to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    loss_function = (
        nn.CrossEntropyLoss()
    )

    training_losses = []

    for epoch in range(
        1,
        selected_epochs + 1,
    ):
        training_loss = train_one_epoch(
            model,
            training_loader,
            optimizer,
            loss_function,
        )

        training_losses.append(
            training_loss
        )

    predictions = predict_data_loader(
        model,
        validation_loader,
    )

    expected_indices = np.flatnonzero(
        masks["outer_validation"]
    )

    assert set(
        predictions["array_index"]
    ) == set(expected_indices)

    prediction_table = pd.DataFrame({
        "array_index": (
            predictions["array_index"]
        ),
        "outer_fold": outer_fold,
        "y_true_index": (
            predictions["y_true"]
        ),
        "y_pred_index": (
            predictions["y_pred"]
        ),
    })

    result = {
        "model_key": model_key,
        "outer_fold": outer_fold,
        "seed": seed,
        "selected_epochs": (
            selected_epochs
        ),
        "final_training_loss": float(
            training_losses[-1]
        ),
        "prediction_table": (
            prediction_table
        ),
    }

    del model
    del optimizer

    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    return result

In [14]:
cnn_outer_results = {}
cnn_prediction_parts = {
    model_key: []
    for model_key in MODEL_CONFIGS
}

selection_rows = []
outer_training_rows = []

for model_key, config in (
    MODEL_CONFIGS.items()
):
    print(f"\n{config['label']}")

    for outer_fold in sorted(
        nested_split_masks
    ):
        cache_key = (
            model_key,
            outer_fold,
            INITIAL_SEED,
        )

        if (
            cache_key
            not in inner_selection_cache
        ):
            inner_selection_cache[
                cache_key
            ] = (
                select_epoch_with_inner_validation(
                    model_key=model_key,
                    outer_fold=outer_fold,
                    seed=INITIAL_SEED,
                    verbose=False,
                )
            )

        selection = (
            inner_selection_cache[
                cache_key
            ]
        )

        print(
            f"  Fold {outer_fold}: "
            f"selected epoch "
            f"{selection['selected_epoch']} "
            f"(inner macro-F1 "
            f"{selection['best_inner_macro_f1']:.4f})"
        )

        outer_result = train_outer_model(
            model_key=model_key,
            outer_fold=outer_fold,
            seed=INITIAL_SEED,
            selected_epochs=(
                selection[
                    "selected_epoch"
                ]
            ),
        )

        cnn_outer_results[
            cache_key
        ] = outer_result

        cnn_prediction_parts[
            model_key
        ].append(
            outer_result[
                "prediction_table"
            ]
        )

        selection_rows.append({
            "model_key": model_key,
            "model": config["label"],
            "outer_fold": outer_fold,
            "inner_validation_fold": (
                selection[
                    "inner_validation_fold"
                ]
            ),
            "selected_epoch": (
                selection[
                    "selected_epoch"
                ]
            ),
            "best_inner_macro_f1": (
                selection[
                    "best_inner_macro_f1"
                ]
            ),
            "epochs_run": (
                selection[
                    "epochs_run"
                ]
            ),
        })

        outer_training_rows.append({
            "model_key": model_key,
            "model": config["label"],
            "outer_fold": outer_fold,
            "selected_epochs": (
                outer_result[
                    "selected_epochs"
                ]
            ),
            "final_training_loss": (
                outer_result[
                    "final_training_loss"
                ]
            ),
        })

cnn_inner_selection_summary = (
    pd.DataFrame(selection_rows)
)

cnn_outer_training_summary = (
    pd.DataFrame(
        outer_training_rows
    )
)

cnn_oof_predictions = {
    model_key: (
        pd.concat(
            prediction_parts,
            ignore_index=True,
        )
        .sort_values("array_index")
        .reset_index(drop=True)
    )
    for model_key, prediction_parts
    in cnn_prediction_parts.items()
}

for predictions in (
    cnn_oof_predictions.values()
):
    assert len(predictions) == len(
        train_metadata
    )
    assert predictions[
        "array_index"
    ].is_unique
    assert np.array_equal(
        predictions[
            "array_index"
        ].to_numpy(),
        np.arange(len(train_metadata)),
    )

display(
    cnn_inner_selection_summary.round(4)
)

print("Complete CNN out-of-fold predictions constructed.")


Accelerometer-only CNN
  Fold 1: selected epoch 15 (inner macro-F1 0.6197)
  Fold 2: selected epoch 28 (inner macro-F1 0.7080)
  Fold 3: selected epoch 10 (inner macro-F1 0.5960)
  Fold 4: selected epoch 26 (inner macro-F1 0.6617)
  Fold 5: selected epoch 28 (inner macro-F1 0.7193)

Dual-branch accelerometer + gyroscope CNN
  Fold 1: selected epoch 16 (inner macro-F1 0.6469)
  Fold 2: selected epoch 27 (inner macro-F1 0.7465)
  Fold 3: selected epoch 16 (inner macro-F1 0.6420)
  Fold 4: selected epoch 12 (inner macro-F1 0.6881)
  Fold 5: selected epoch 24 (inner macro-F1 0.7551)


,model_key,model,outer_fold,inner_validation_fold,selected_epoch,best_inner_macro_f1,epochs_run
0,accelerometer,Accelerometer-only CNN,1,2,15,0.6197,21
1,accelerometer,Accelerometer-only CNN,2,3,28,0.7080,34
2,accelerometer,Accelerometer-only CNN,3,4,10,0.5960,16
3,accelerometer,Accelerometer-only CNN,4,5,26,0.6617,32
4,accelerometer,Accelerometer-only CNN,5,1,28,0.7193,34
5,dual_branch,Dual-branch accelerometer + gyroscope CNN,1,2,16,0.6469,22
6,dual_branch,Dual-branch accelerometer + gyroscope CNN,2,3,27,0.7465,33
7,dual_branch,Dual-branch accelerometer + gyroscope CNN,3,4,16,0.6420,22
8,dual_branch,Dual-branch accelerometer + gyroscope CNN,4,5,12,0.6881,18
9,dual_branch,Dual-branch accelerometer + gyroscope CNN,5,1,24,0.7551,30


Complete CNN out-of-fold predictions constructed.


Evaluate the two models:

In [15]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
)


ENGINEERED_RF_FOLD_MACRO_F1 = {
    1: 0.7673,
    2: 0.6634,
    3: 0.7675,
    4: 0.6992,
    5: 0.7310,
}

CNN_RF_IMPROVEMENT_TARGET = 0.7357


def calculate_cnn_metrics(
    y_true,
    y_pred,
):
    class_indices = np.arange(
        len(ACTIVITY_CODES)
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": recall_score(
            y_true,
            y_pred,
            labels=class_indices,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=class_indices,
            average="macro",
            zero_division=0,
        ),
    }


cnn_fold_rows = []

for model_key, predictions in (
    cnn_oof_predictions.items()
):
    for outer_fold, fold_predictions in (
        predictions.groupby(
            "outer_fold",
            sort=True,
        )
    ):
        cnn_fold_rows.append({
            "model_key": model_key,
            "model": MODEL_CONFIGS[
                model_key
            ]["label"],
            "fold": int(outer_fold),
            **calculate_cnn_metrics(
                fold_predictions[
                    "y_true_index"
                ],
                fold_predictions[
                    "y_pred_index"
                ],
            ),
        })

cnn_fold_metrics = pd.DataFrame(
    cnn_fold_rows
)

cnn_fold_macro_f1 = (
    cnn_fold_metrics
    .pivot(
        index="fold",
        columns="model_key",
        values="macro_f1",
    )
)

cnn_fold_macro_f1[
    "engineered_rf"
] = pd.Series(
    ENGINEERED_RF_FOLD_MACRO_F1
)

cnn_summary_rows = []

for model_key, config in (
    MODEL_CONFIGS.items()
):
    model_metrics = (
        cnn_fold_metrics.loc[
            cnn_fold_metrics[
                "model_key"
            ].eq(model_key)
        ]
    )

    summary = (
        model_metrics[
            [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
            ]
        ]
        .agg(["mean", "std"])
    )

    folds_better_than_rf = int(
        (
            cnn_fold_macro_f1[
                model_key
            ]
            > cnn_fold_macro_f1[
                "engineered_rf"
            ]
        ).sum()
    )

    cnn_summary_rows.append({
        "model_key": model_key,
        "model": config["label"],
        "accuracy_mean": summary.loc[
            "mean",
            "accuracy",
        ],
        "accuracy_std": summary.loc[
            "std",
            "accuracy",
        ],
        "balanced_accuracy_mean": (
            summary.loc[
                "mean",
                "balanced_accuracy",
            ]
        ),
        "balanced_accuracy_std": (
            summary.loc[
                "std",
                "balanced_accuracy",
            ]
        ),
        "macro_f1_mean": summary.loc[
            "mean",
            "macro_f1",
        ],
        "macro_f1_std": summary.loc[
            "std",
            "macro_f1",
        ],
        "folds_better_than_rf": (
            folds_better_than_rf
        ),
        "passes_rf_improvement_gate": (
            summary.loc[
                "mean",
                "macro_f1",
            ]
            >= CNN_RF_IMPROVEMENT_TARGET
            and folds_better_than_rf >= 3
        ),
    })

cnn_summary = pd.DataFrame(
    cnn_summary_rows
)

dual_gain_by_fold = (
    cnn_fold_macro_f1[
        "dual_branch"
    ]
    - cnn_fold_macro_f1[
        "accelerometer"
    ]
)

dual_mean_gain = (
    dual_gain_by_fold.mean()
)

dual_folds_improved = int(
    dual_gain_by_fold.gt(0).sum()
)

display(
    cnn_fold_macro_f1.round(4)
)

display(
    cnn_summary.round(4)
)

print(
    "Dual-branch mean gain:",
    f"{dual_mean_gain:.4f}",
)
print(
    "Folds improved by dual branch:",
    dual_folds_improved,
)
print(
    "Dual-branch modality gate:",
    (
        "PASS"
        if (
            dual_mean_gain >= 0.01
            and dual_folds_improved >= 3
        )
        else "FAIL"
    ),
)

model_key,accelerometer,dual_branch,engineered_rf
fold,,,
1,0.6874,0.7472,0.7673
2,0.6271,0.6158,0.6634
3,0.7029,0.7239,0.7675
4,0.6127,0.6145,0.6992
5,0.6378,0.6783,0.7310


,model_key,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,folds_better_than_rf,passes_rf_improvement_gate
0,accelerometer,Accelerometer-only CNN,0.6544,0.0428,0.6561,0.0425,0.6536,0.0394,0,False
1,dual_branch,Dual-branch accelerometer + gyroscope CNN,0.6766,0.0638,0.6778,0.0629,0.6760,0.0608,0,False


Dual-branch mean gain: 0.0224
Folds improved by dual branch: 4
Dual-branch modality gate: PASS


## 6. Stability of the selected CNN across random seeds

The seed-42 comparison selected the dual-branch accelerometer–gyroscope
CNN as the stronger neural architecture:

- it improved mean macro-F1 by more than 0.01 over the accelerometer-only
  CNN;
- it improved 4 of the 5 outer folds.

However, it did not outperform the engineered Random Forest on any outer
fold. Before drawing the final conclusion, the selected dual-branch
architecture is repeated with seeds 43 and 44, in addition to the completed
seed-42 run.

For every seed, inner epoch selection and outer-model refitting are repeated.
The architecture, optimizer, learning rate, early-stopping rule, folds, and
all other settings remain unchanged.

In [17]:
STABILITY_SEEDS = [42, 43, 44]

dual_seed_prediction_parts = [
    cnn_oof_predictions[
        "dual_branch"
    ].assign(seed=INITIAL_SEED)
]

seed_42_selection = (
    cnn_inner_selection_summary
    .loc[
        cnn_inner_selection_summary[
            "model_key"
        ].eq("dual_branch"),
        [
            "outer_fold",
            "inner_validation_fold",
            "selected_epoch",
            "best_inner_macro_f1",
            "epochs_run",
        ],
    ]
    .assign(seed=INITIAL_SEED)
)

dual_seed_selection_parts = [
    seed_42_selection
]

for seed in STABILITY_SEEDS:
    if seed == INITIAL_SEED:
        continue

    seed_prediction_parts = []
    seed_selection_rows = []

    print(f"\nSeed {seed}")

    for outer_fold in sorted(
        nested_split_masks
    ):
        selection = (
            select_epoch_with_inner_validation(
                model_key="dual_branch",
                outer_fold=outer_fold,
                seed=seed,
                verbose=False,
            )
        )

        cache_key = (
            "dual_branch",
            outer_fold,
            seed,
        )

        inner_selection_cache[
            cache_key
        ] = selection

        outer_result = train_outer_model(
            model_key="dual_branch",
            outer_fold=outer_fold,
            seed=seed,
            selected_epochs=(
                selection[
                    "selected_epoch"
                ]
            ),
        )

        cnn_outer_results[
            cache_key
        ] = outer_result

        fold_predictions = (
            outer_result[
                "prediction_table"
            ].copy()
        )
        fold_predictions["seed"] = seed

        seed_prediction_parts.append(
            fold_predictions
        )

        seed_selection_rows.append({
            "seed": seed,
            "outer_fold": outer_fold,
            "inner_validation_fold": (
                selection[
                    "inner_validation_fold"
                ]
            ),
            "selected_epoch": (
                selection[
                    "selected_epoch"
                ]
            ),
            "best_inner_macro_f1": (
                selection[
                    "best_inner_macro_f1"
                ]
            ),
            "epochs_run": (
                selection[
                    "epochs_run"
                ]
            ),
        })

        print(
            f"  Fold {outer_fold}: "
            f"selected epoch "
            f"{selection['selected_epoch']} "
            f"(inner macro-F1 "
            f"{selection['best_inner_macro_f1']:.4f})"
        )

    seed_predictions = (
        pd.concat(
            seed_prediction_parts,
            ignore_index=True,
        )
        .sort_values("array_index")
        .reset_index(drop=True)
    )

    assert len(seed_predictions) == len(
        train_metadata
    )
    assert seed_predictions[
        "array_index"
    ].is_unique
    assert np.array_equal(
        seed_predictions[
            "array_index"
        ].to_numpy(),
        np.arange(len(train_metadata)),
    )

    dual_seed_prediction_parts.append(
        seed_predictions
    )
    dual_seed_selection_parts.append(
        pd.DataFrame(
            seed_selection_rows
        )
    )

dual_seed_oof_predictions = (
    pd.concat(
        dual_seed_prediction_parts,
        ignore_index=True,
    )
    .sort_values(
        ["seed", "array_index"]
    )
    .reset_index(drop=True)
)

dual_seed_selection_summary = (
    pd.concat(
        dual_seed_selection_parts,
        ignore_index=True,
    )
    .sort_values(
        ["seed", "outer_fold"]
    )
    .reset_index(drop=True)
)

assert len(
    dual_seed_oof_predictions
) == (
    len(STABILITY_SEEDS)
    * len(train_metadata)
)

display(
    dual_seed_selection_summary.round(4)
)


Seed 43
  Fold 1: selected epoch 7 (inner macro-F1 0.6308)
  Fold 2: selected epoch 14 (inner macro-F1 0.7454)
  Fold 3: selected epoch 14 (inner macro-F1 0.6459)
  Fold 4: selected epoch 11 (inner macro-F1 0.6899)
  Fold 5: selected epoch 4 (inner macro-F1 0.7409)

Seed 44
  Fold 1: selected epoch 9 (inner macro-F1 0.6346)
  Fold 2: selected epoch 22 (inner macro-F1 0.7543)
  Fold 3: selected epoch 26 (inner macro-F1 0.6419)
  Fold 4: selected epoch 5 (inner macro-F1 0.6807)
  Fold 5: selected epoch 4 (inner macro-F1 0.7351)


,outer_fold,inner_validation_fold,selected_epoch,best_inner_macro_f1,epochs_run,seed
0,1,2,16,0.6469,22,42
1,2,3,27,0.7465,33,42
2,3,4,16,0.6420,22,42
3,4,5,12,0.6881,18,42
4,5,1,24,0.7551,30,42
5,1,2,7,0.6308,13,43
6,2,3,14,0.7454,20,43
7,3,4,14,0.6459,20,43
8,4,5,11,0.6899,17,43
9,5,1,4,0.7409,10,43


### Three-seed outer-fold evaluation

Selected training durations vary across seeds, while the corresponding inner
macro-F1 values remain broadly similar. The relevant stability question is
therefore whether the independently refitted models produce consistent
outer-fold performance.

Metrics are calculated separately for every seed and outer fold. The primary
three-seed result is the mean of the five-fold macro-F1 for each seed, followed
by the mean and standard deviation of those three seed-level results.

In [18]:
dual_seed_fold_rows = []

for (
    seed,
    outer_fold,
), fold_predictions in (
    dual_seed_oof_predictions
    .groupby(
        ["seed", "outer_fold"],
        sort=True,
    )
):
    dual_seed_fold_rows.append({
        "seed": int(seed),
        "outer_fold": int(
            outer_fold
        ),
        **calculate_cnn_metrics(
            fold_predictions[
                "y_true_index"
            ],
            fold_predictions[
                "y_pred_index"
            ],
        ),
    })

dual_seed_fold_metrics = (
    pd.DataFrame(
        dual_seed_fold_rows
    )
)

dual_seed_summary = (
    dual_seed_fold_metrics
    .groupby("seed")
    .agg(
        accuracy_mean=(
            "accuracy",
            "mean",
        ),
        accuracy_fold_std=(
            "accuracy",
            "std",
        ),
        balanced_accuracy_mean=(
            "balanced_accuracy",
            "mean",
        ),
        balanced_accuracy_fold_std=(
            "balanced_accuracy",
            "std",
        ),
        macro_f1_mean=(
            "macro_f1",
            "mean",
        ),
        macro_f1_fold_std=(
            "macro_f1",
            "std",
        ),
    )
    .reset_index()
)

dual_seed_fold_comparison = (
    dual_seed_fold_metrics
    .pivot(
        index="outer_fold",
        columns="seed",
        values="macro_f1",
    )
)

dual_seed_fold_comparison.columns = [
    f"seed_{seed}"
    for seed
    in dual_seed_fold_comparison.columns
]

seed_columns = [
    f"seed_{seed}"
    for seed in STABILITY_SEEDS
]

dual_seed_fold_comparison[
    "cnn_seed_mean"
] = (
    dual_seed_fold_comparison[
        seed_columns
    ].mean(axis=1)
)

dual_seed_fold_comparison[
    "cnn_seed_std"
] = (
    dual_seed_fold_comparison[
        seed_columns
    ].std(axis=1)
)

dual_seed_fold_comparison[
    "engineered_rf"
] = pd.Series(
    ENGINEERED_RF_FOLD_MACRO_F1
)

dual_seed_fold_comparison[
    "cnn_mean_minus_rf"
] = (
    dual_seed_fold_comparison[
        "cnn_seed_mean"
    ]
    - dual_seed_fold_comparison[
        "engineered_rf"
    ]
)

three_seed_macro_f1_mean = (
    dual_seed_summary[
        "macro_f1_mean"
    ].mean()
)

three_seed_macro_f1_std = (
    dual_seed_summary[
        "macro_f1_mean"
    ].std()
)

folds_better_than_rf = int(
    dual_seed_fold_comparison[
        "cnn_mean_minus_rf"
    ].gt(0).sum()
)

passes_rf_improvement_gate = (
    three_seed_macro_f1_mean
    >= CNN_RF_IMPROVEMENT_TARGET
    and folds_better_than_rf >= 3
)

display(
    dual_seed_fold_metrics.round(4)
)

display(
    dual_seed_summary.round(4)
)

display(
    dual_seed_fold_comparison.round(4)
)

print(
    "Three-seed mean macro-F1:",
    f"{three_seed_macro_f1_mean:.4f}",
)
print(
    "Std across seed means:",
    f"{three_seed_macro_f1_std:.4f}",
)
print(
    "Engineered RF mean macro-F1:",
    "0.7257",
)
print(
    "Difference from engineered RF:",
    f"{three_seed_macro_f1_mean - 0.7257:+.4f}",
)
print(
    "Folds where mean CNN beats RF:",
    f"{folds_better_than_rf}/5",
)
print(
    "CNN-versus-RF gate:",
    (
        "PASS"
        if passes_rf_improvement_gate
        else "FAIL"
    ),
)

,seed,outer_fold,accuracy,balanced_accuracy,macro_f1
0,42,1,0.7504,0.7504,0.7472
1,42,2,0.6171,0.6184,0.6158
2,42,3,0.7286,0.7290,0.7239
3,42,4,0.6091,0.6114,0.6145
4,42,5,0.6779,0.6796,0.6783
5,43,1,0.7508,0.7508,0.7469
6,43,2,0.6298,0.6318,0.6282
7,43,3,0.7306,0.7326,0.7269
8,43,4,0.6187,0.6204,0.6235
9,43,5,0.6663,0.6683,0.6611


,seed,accuracy_mean,accuracy_fold_std,balanced_accuracy_mean,balanced_accuracy_fold_std,macro_f1_mean,macro_f1_fold_std
0,42,0.6766,0.0638,0.6778,0.0629,0.6760,0.0608
1,43,0.6792,0.0592,0.6808,0.0587,0.6773,0.0567
2,44,0.6765,0.0695,0.6776,0.0696,0.6753,0.0649


,seed_42,seed_43,seed_44,cnn_seed_mean,cnn_seed_std,engineered_rf,cnn_mean_minus_rf
outer_fold,,,,,,,
1,0.7472,0.7469,0.7448,0.7463,0.0013,0.7673,-0.0210
2,0.6158,0.6282,0.6370,0.6270,0.0106,0.6634,-0.0364
3,0.7239,0.7269,0.7344,0.7284,0.0054,0.7675,-0.0391
4,0.6145,0.6235,0.5914,0.6098,0.0166,0.6992,-0.0894
5,0.6783,0.6611,0.6690,0.6695,0.0086,0.7310,-0.0615


Three-seed mean macro-F1: 0.6762
Std across seed means: 0.0010
Engineered RF mean macro-F1: 0.7257
Difference from engineered RF: -0.0495
Folds where mean CNN beats RF: 0/5
CNN-versus-RF gate: FAIL


## 7. Findings and decision

The seed-42 architecture comparison showed that combining watch
accelerometer and gyroscope signals was beneficial:

- the dual-branch CNN improved mean macro-F1 from 0.6536 to 0.6760;
- it improved 4 of the 5 outer folds;
- it therefore passed the predeclared modality-fusion gate.

The selected dual-branch CNN was then evaluated with three complete training
runs. Its seed-level mean macro-F1 values were 0.6760, 0.6773, and 0.6753,
giving an overall mean of **0.6762** and a standard deviation of only
**0.0010 across seed means**. Although inner-selected training durations varied,
the resulting development performance was highly stable across random seeds.

The CNN remained below the engineered Random Forest on every outer fold. Its
three-seed mean was **0.0495 macro-F1 lower** than the RF result of 0.7257, so
it failed the predeclared CNN-versus-RF improvement gate. Performance varied
much more between subject folds than between random seeds, with fold 4
remaining the most difficult split.

These results support two conclusions:

1. Gyroscope information provides useful complementary information to the
   accelerometer signal.
2. Under the fixed architecture and training protocol tested here, direct
   temporal learning with this small CNN does not outperform the engineered
   statistical and frequency features used by the Random Forest.

The engineered Random Forest therefore remains the strongest development
model. This result does not establish that every neural architecture would
underperform, but it does not justify adding neural complexity based on the
current experiment. The reserved test set remains untouched.